# Bulgarian School Exam Results — НВО & ДЗИ

**Dataset by [Ivan Davidov](https://ivandavidov.github.io/nvo)**

This notebook explores national exam results for all schools in Bulgaria:
- **НВО** (National External Assessment) — grades 4, 7, 10
- **ДЗИ** (State Matriculation Exams) — grade 12

We'll look at trends over time, compare cities and school types, and find the top-performing schools.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load the data

In [ ]:
import os

# Adjust path for Kaggle environment
BASE = next(
    os.path.join(root, '')
    for root, dirs, files in os.walk('/kaggle/input')
    if 'scores.csv' in files
)
print(f'Data path: {BASE}')

In [ ]:
scores = pd.read_csv(BASE + 'scores.csv')
rankings = pd.read_csv(BASE + 'rankings.csv')
schools = pd.read_csv(BASE + 'schools.csv', dtype={'code': str})
cities = pd.read_csv(BASE + 'cities.csv')

scores['school_code'] = scores['school_code'].astype(str)
rankings['school_code'] = rankings['school_code'].astype(str)
schools['code'] = schools['code'].astype(str)

print(f'Scores:   {scores.shape[0]:,} rows')
print(f'Rankings: {rankings.shape[0]:,} rows')
print(f'Schools:  {schools.shape[0]:,} rows')
print(f'Cities:   {cities.shape[0]:,} rows')

In [ ]:
scores.head(10)

## 2. Data overview

In [ ]:
summary = scores.groupby(['grade', 'year']).agg(
    schools_count=('school_code', 'nunique'),
    avg_bel=('bel_score', 'mean'),
    avg_mat=('mat_score', 'mean'),
    total_bel_students=('bel_students', 'sum'),
    total_mat_students=('mat_students', 'sum')
).round(2)

summary

## 3. National average scores over time

How have average BEL (Bulgarian Language) and MAT (Mathematics) scores changed across the years?

In [ ]:
grade_labels = {4: 'Grade 4 (НВО)', 7: 'Grade 7 (НВО)', 10: 'Grade 10 (НВО)', 12: 'Grade 12 (ДЗИ)'}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, grade in zip(axes.flat, [4, 7, 10, 12]):
    gdata = scores[scores['grade'] == grade].groupby('year')[['bel_score', 'mat_score']].mean()
    gdata.plot(ax=ax, marker='o', linewidth=2)
    ax.set_title(grade_labels[grade], fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Average Score')
    ax.legend(['BEL', 'MAT'])
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.suptitle('National Average Scores by Grade (2018–2025)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Number of participating schools over time

In [ ]:
participation = scores.groupby(['grade', 'year'])['school_code'].nunique().reset_index()
participation.columns = ['grade', 'year', 'num_schools']

fig, ax = plt.subplots(figsize=(12, 5))
for grade in [4, 7, 10, 12]:
    gdata = participation[participation['grade'] == grade]
    ax.plot(gdata['year'], gdata['num_schools'], marker='o', linewidth=2, label=grade_labels[grade])

ax.set_title('Number of Participating Schools per Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Schools')
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 5. Top 10 schools — Grade 7 (2025)

The most competitive grade in Bulgaria — entry exams for elite high schools.

In [ ]:
top10 = rankings[
    (rankings['grade'] == 7) & (rankings['year'] == 2025) & (rankings['type'] == 'year')
].head(10).merge(schools[['code', 'full_name', 'is_private']], left_on='school_code', right_on='code')

top10[['rank', 'full_name', 'city', 'bel_score', 'mat_score', 'score', 'is_private']]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_pos = range(len(top10))
bar_h = 0.35

ax.barh([y - bar_h/2 for y in y_pos], top10['bel_score'], bar_h, label='BEL', color='#3884F4')
ax.barh([y + bar_h/2 for y in y_pos], top10['mat_score'], bar_h, label='MAT', color='#F6AD42')

ax.set_yticks(y_pos)
ax.set_yticklabels(top10['full_name'])
ax.invert_yaxis()
ax.set_xlabel('Score')
ax.set_title('Top 10 Schools — Grade 7, 2025', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Public vs. Private schools

In [ ]:
merged = scores.merge(schools[['code', 'is_private']], left_on='school_code', right_on='code')

comparison = merged.groupby(['grade', 'year', 'is_private']).agg(
    avg_bel=('bel_score', 'mean'),
    avg_mat=('mat_score', 'mean'),
    num_schools=('school_code', 'nunique')
).round(2).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, grade in zip(axes.flat, [4, 7, 10, 12]):
    for priv, style, label in [(False, '-', 'Public'), (True, '--', 'Private')]:
        d = comparison[(comparison['grade'] == grade) & (comparison['is_private'] == priv)]
        avg = (d['avg_bel'] + d['avg_mat']) / 2
        ax.plot(d['year'], avg, style, marker='o', linewidth=2, label=label)
    ax.set_title(grade_labels[grade], fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Avg Score (BEL+MAT)/2')
    ax.legend()
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.suptitle('Public vs. Private School Performance', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Top 10 cities by average score — Grade 7 (2025)

In [ ]:
city_avg = scores[(scores['grade'] == 7) & (scores['year'] == 2025)].groupby('city').agg(
    avg_bel=('bel_score', 'mean'),
    avg_mat=('mat_score', 'mean'),
    num_schools=('school_code', 'nunique')
).round(2)

city_avg['avg_total'] = ((city_avg['avg_bel'] + city_avg['avg_mat']) / 2).round(2)
city_avg = city_avg.sort_values('avg_total', ascending=False)

# Merge with city names
top_cities = city_avg.head(10).reset_index().merge(cities[['slug', 'full_name']], left_on='city', right_on='slug')

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(top_cities))
w = 0.35
ax.bar([i - w/2 for i in x], top_cities['avg_bel'], w, label='BEL', color='#3884F4')
ax.bar([i + w/2 for i in x], top_cities['avg_mat'], w, label='MAT', color='#F6AD42')
ax.set_xticks(x)
ax.set_xticklabels(top_cities['full_name'], rotation=30, ha='right')
ax.set_ylabel('Average Score')
ax.set_title('Top 10 Cities by Average Score — Grade 7, 2025', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Score distribution — Grade 7 (2025)

In [ ]:
g7_2025 = scores[(scores['grade'] == 7) & (scores['year'] == 2025)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(g7_2025['bel_score'].dropna(), bins=30, color='#3884F4', edgecolor='white', alpha=0.85)
axes[0].set_title('BEL Score Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Number of Schools')
axes[0].axvline(g7_2025['bel_score'].mean(), color='red', linestyle='--', label=f'Mean: {g7_2025["bel_score"].mean():.1f}')
axes[0].legend()

axes[1].hist(g7_2025['mat_score'].dropna(), bins=30, color='#F6AD42', edgecolor='white', alpha=0.85)
axes[1].set_title('MAT Score Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Number of Schools')
axes[1].axvline(g7_2025['mat_score'].mean(), color='red', linestyle='--', label=f'Mean: {g7_2025["mat_score"].mean():.1f}')
axes[1].legend()

fig.suptitle('Score Distribution — Grade 7, 2025', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. BEL vs. MAT correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, grade, title in [(axes[0], 7, 'Grade 7 (НВО)'), (axes[1], 12, 'Grade 12 (ДЗИ)')]:
    gd = scores[(scores['grade'] == grade) & (scores['year'] == 2025)].dropna(subset=['bel_score', 'mat_score'])
    ax.scatter(gd['bel_score'], gd['mat_score'], alpha=0.4, s=20, color='#3884F4')
    ax.set_xlabel('BEL Score')
    ax.set_ylabel('MAT Score')
    ax.set_title(f'{title} — BEL vs MAT (2025)', fontsize=13, fontweight='bold')
    corr = gd[['bel_score', 'mat_score']].corr().iloc[0, 1]
    ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes, fontsize=12,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

---

## About this dataset

Created and maintained by **[Ivan Davidov](https://www.linkedin.com/in/ivandavidov)**.

- **Website:** https://ivandavidov.github.io/nvo
- **JSON API:** https://ivandavidov.github.io/nvo/api/v1/
- **GitHub:** https://github.com/ivandavidov/nvo

Raw data sourced from the Bulgarian Open Data Portal ([data.egov.bg](https://data.egov.bg)).